# Federated Learning with Flower

Flower (flwr) is a production federated learning framework that handles client management, communication, and simulation. This notebook implements a 10-client federated simulation using Flower-compatible patterns.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import random, math

# Flower-compatible interface (without importing flwr)
@dataclass
class FitIns:
    parameters: list
    config: dict

@dataclass
class FitRes:
    parameters: list
    num_examples: int
    metrics: dict

@dataclass
class EvaluateIns:
    parameters: list
    config: dict

@dataclass
class EvaluateRes:
    loss: float
    num_examples: int
    metrics: dict

class FlowerClient:
    def __init__(self, cid: int, local_data: list):
        self.cid = cid
        self.data = local_data
        self.model = [0.0]

    def get_parameters(self) -> list:
        return list(self.model)

    def fit(self, ins: FitIns) -> FitRes:
        self.model = list(ins.parameters)
        lr = ins.config.get('lr', 0.01)
        epochs = ins.config.get('epochs', 5)
        rng = random.Random(self.cid)
        for _ in range(epochs):
            batch = rng.sample(self.data, min(32, len(self.data)))
            grad = sum(2*(self.model[0]-x) for x in batch) / len(batch)
            self.model[0] -= lr * grad
        loss = sum((self.model[0]-x)**2 for x in self.data) / len(self.data)
        return FitRes(parameters=list(self.model), num_examples=len(self.data),
                       metrics={'local_loss': loss})

    def evaluate(self, ins: EvaluateIns) -> EvaluateRes:
        model = ins.parameters
        loss = sum((model[0]-x)**2 for x in self.data) / len(self.data)
        return EvaluateRes(loss=loss, num_examples=len(self.data), metrics={'mse': loss})

class FedAvgStrategy:
    def __init__(self, fraction_fit=0.5, fraction_evaluate=1.0,
                 min_fit_clients=2, min_evaluate_clients=2):
        self.fraction_fit = fraction_fit
        self.fraction_evaluate = fraction_evaluate
        self.min_fit = min_fit_clients
        self.min_evaluate = min_evaluate_clients

    def configure_fit(self, server_round: int, parameters: list, client_ids: list) -> list:
        n = max(self.min_fit, int(len(client_ids)*self.fraction_fit))
        selected = random.sample(client_ids, min(n, len(client_ids)))
        config = {'lr': 0.01, 'epochs': 5, 'round': server_round}
        return [(cid, FitIns(parameters=list(parameters), config=config)) for cid in selected]

    def aggregate_fit(self, results: list) -> list:
        total = sum(r.num_examples for _, r in results)
        return [sum(r.parameters[i]*r.num_examples for _, r in results)/total
                for i in range(len(results[0][1].parameters))]

    def aggregate_evaluate(self, results: list) -> float:
        total = sum(r.num_examples for _, r in results)
        return sum(r.loss*r.num_examples for _, r in results) / total

class FlowerSimulation:
    def __init__(self, clients: list, strategy: FedAvgStrategy, n_rounds: int = 10):
        self.clients = {c.cid: c for c in clients}
        self.strategy = strategy
        self.n_rounds = n_rounds
        self.parameters = [0.0]
        self.history = []

    def run(self) -> list:
        for r in range(1, self.n_rounds+1):
            fit_configs = self.strategy.configure_fit(r, self.parameters, list(self.clients.keys()))
            fit_results = [(cid, self.clients[cid].fit(ins)) for cid, ins in fit_configs]
            self.parameters = self.strategy.aggregate_fit(fit_results)
            eval_results = [(cid, c.evaluate(EvaluateIns(self.parameters, {})))
                             for cid, c in self.clients.items()]
            agg_loss = self.strategy.aggregate_evaluate(eval_results)
            self.history.append({'round': r, 'model': self.parameters[0],
                                  'loss': agg_loss, 'n_fit': len(fit_configs)})
        return self.history

# 10 clients with heterogeneous data
rng = random.Random(42)
clients = [FlowerClient(i, [rng.gauss(float(i+1), 0.5) for _ in range(200)]) for i in range(10)]
strategy = FedAvgStrategy(fraction_fit=0.5, min_fit_clients=3)
sim = FlowerSimulation(clients, strategy, n_rounds=15)
history = sim.run()

print(f'Flower FL Simulation: 10 clients, 15 rounds')
print(f'{'Round':>7} {'Model':>10} {'Loss':>10} {'Clients':>9}')
for h in history[::3]:
    print(f"{h['round']:>7} {h['model']:>10.4f} {h['loss']:>10.6f} {h['n_fit']:>9}")
print(f'\nTrue global mean: 5.5')
print(f'Final model: {history[-1]["model"]:.4f}')